# Notebook 6 — Train, Tune, Evaluate

**Rules:**
- Start with a simple baseline so you know what you have to beat.
- Tune using the validation split only.
- Choose a metric that fits an imbalanced problem — not accuracy alone.
- Touch the test set exactly once, at the very end.

In [1]:
import pandas as pd
import numpy as np
import pickle

ARTIFACTS_DIR = "artifacts"

train_df = pd.read_parquet(f"{ARTIFACTS_DIR}/05_train_features.parquet")
val_df   = pd.read_parquet(f"{ARTIFACTS_DIR}/05_val_features.parquet")
test_df  = pd.read_parquet(f"{ARTIFACTS_DIR}/05_test_features.parquet")

with open(f"{ARTIFACTS_DIR}/05_feature_list.pkl", "rb") as f:
    FEATURE_LIST = pickle.load(f)

X_train, y_train = train_df[FEATURE_LIST], train_df['is_late']
X_val, y_val     = val_df[FEATURE_LIST],   val_df['is_late']
X_test, y_test   = test_df[FEATURE_LIST],  test_df['is_late']

print(X_train.shape, X_val.shape, X_test.shape)
print("Late rate — train/val/test:", y_train.mean(), y_val.mean(), y_test.mean())

(67533, 119) (14471, 119) (14472, 119)
Late rate — train/val/test: 0.0902817881628241 0.053417179185958126 0.06612769485903815


## Baseline: what you have to beat
A dummy classifier that always predicts the majority class ("never late")
already gets high *accuracy* on an imbalanced dataset — which is exactly why
accuracy alone is the wrong metric here.

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix)

baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_val)
base_proba = baseline.predict_proba(X_val)[:, 1]

print("BASELINE (val):")
print(f"  accuracy : {accuracy_score(y_val, base_pred):.4f}")
print(f"  f1       : {f1_score(y_val, base_pred, zero_division=0):.4f}")
print(f"  roc_auc  : {roc_auc_score(y_val, base_proba):.4f}")
print(f"  PR-auc   : {average_precision_score(y_val, base_proba):.4f}")

BASELINE (val):
  accuracy : 0.9466
  f1       : 0.0000
  roc_auc  : 0.5000
  PR-auc   : 0.0534


## Metric choice
For an imbalanced late/on-time problem, **PR-AUC (average precision)** and
**F1 on the minority (late) class** are more informative than accuracy or even
plain ROC-AUC, because they focus on how well the model finds the rare "late"
cases rather than being flattered by the easy majority class. We'll track
these primarily and report accuracy/ROC-AUC as secondary context.

In [3]:
def evaluate(model, X, y, name="model"):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    print(f"--- {name} ---")
    print(f"  accuracy : {accuracy_score(y, pred):.4f}")
    print(f"  precision: {precision_score(y, pred, zero_division=0):.4f}")
    print(f"  recall   : {recall_score(y, pred, zero_division=0):.4f}")
    print(f"  f1       : {f1_score(y, pred, zero_division=0):.4f}")
    print(f"  roc_auc  : {roc_auc_score(y, proba):.4f}")
    print(f"  pr_auc   : {average_precision_score(y, proba):.4f}")
    print(confusion_matrix(y, pred))
    return {"accuracy": accuracy_score(y, pred), "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, proba), "pr_auc": average_precision_score(y, proba)}

## Train a real model (Logistic Regression + Random Forest) and tune on validation

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)
logreg_metrics = evaluate(logreg, X_val, y_val, "LogisticRegression (val)")

--- LogisticRegression (val) ---
  accuracy : 0.7602
  precision: 0.1480
  recall   : 0.7335
  f1       : 0.2463
  roc_auc  : 0.8283
  pr_auc   : 0.2660
[[10434  3264]
 [  206   567]]


In [5]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_metrics = evaluate(rf, X_val, y_val, "RandomForest baseline (val)")

--- RandomForest baseline (val) ---
  accuracy : 0.9480
  precision: 0.6296
  recall   : 0.0660
  f1       : 0.1194
  roc_auc  : 0.7780
  pr_auc   : 0.2454
[[13668    30]
 [  722    51]]


## Simple tuning on validation (grid over a few RandomForest params)

In [6]:
from itertools import product

best_score, best_params, best_model = -1, None, None
for n_estimators, max_depth in product([200, 400], [8, 16, None]):
    m = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    m.fit(X_train, y_train)
    proba = m.predict_proba(X_val)[:, 1]
    score = average_precision_score(y_val, proba)  # our chosen metric
    print(f"n_estimators={n_estimators:4d} max_depth={str(max_depth):5s} -> PR-AUC={score:.4f}")
    if score > best_score:
        best_score, best_params, best_model = score, (n_estimators, max_depth), m

print("\nBest params:", best_params, "PR-AUC:", best_score)

n_estimators= 200 max_depth=8     -> PR-AUC=0.1977
n_estimators= 200 max_depth=16    -> PR-AUC=0.2474
n_estimators= 200 max_depth=None  -> PR-AUC=0.2409
n_estimators= 400 max_depth=8     -> PR-AUC=0.1977
n_estimators= 400 max_depth=16    -> PR-AUC=0.2528
n_estimators= 400 max_depth=None  -> PR-AUC=0.2471

Best params: (400, 16) PR-AUC: 0.25284401139973794


In [7]:
final_model = best_model
val_metrics = evaluate(final_model, X_val, y_val, "FINAL MODEL (val)")

--- FINAL MODEL (val) ---
  accuracy : 0.9335
  precision: 0.3395
  recall   : 0.2600
  f1       : 0.2945
  roc_auc  : 0.7759
  pr_auc   : 0.2528
[[13307   391]
 [  572   201]]


## Touch the test set ONCE, at the very end

In [8]:
test_metrics = evaluate(final_model, X_test, y_test, "FINAL MODEL (test — touched once)")

--- FINAL MODEL (test — touched once) ---
  accuracy : 0.9087
  precision: 0.2759
  recall   : 0.2341
  f1       : 0.2533
  roc_auc  : 0.6884
  pr_auc   : 0.2377
[[12927   588]
 [  733   224]]


## Artifact: the trained model and a results summary

In [9]:
import json

with open(f"{ARTIFACTS_DIR}/06_final_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

results_summary = {
    "baseline_val": {"strategy": "most_frequent"},
    "best_params": {"n_estimators": best_params[0], "max_depth": best_params[1]},
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
}
with open(f"{ARTIFACTS_DIR}/06_results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)

print(json.dumps(results_summary, indent=2, default=str))
print("\nSaved model + results summary to", ARTIFACTS_DIR)

{
  "baseline_val": {
    "strategy": "most_frequent"
  },
  "best_params": {
    "n_estimators": 400,
    "max_depth": 16
  },
  "val_metrics": {
    "accuracy": 0.9334531131227973,
    "f1": 0.2945054945054945,
    "roc_auc": 0.7759180337560728,
    "pr_auc": 0.25284401139973794
  },
  "test_metrics": {
    "accuracy": 0.9087202874516307,
    "f1": 0.2532504239683437,
    "roc_auc": 0.6884118462747573,
    "pr_auc": 0.2377042826103563
  }
}

Saved model + results summary to artifacts
